# Smartphone Battery Drain Simulation
This notebook simulates battery drain based on the Continuous-Time mathematical model developed in the project report. It utilizes the **2RC-Thevenin Equivalent Circuit Model** and physical power laws for CPU, Display, and Network drain.

In [ ]:
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt

# Battery Constants
Q_CAP_MAX = 3274 * 3600  # mAh to Coulomb (As)
R_INTERNAL = 0.0233      # Ohms
V_NOMINAL = 3.7          # Volts
TAU_TAIL = 1.5           # Seconds (Tail Duration)

## 1. Battery Dynamics Model
The `SmartphoneBattery` class encapsulates the physical parameters and the OCV curve.

In [ ]:
class SmartphoneBattery:
    def __init__(self, soc_init=1.0, health=1.0):
        self.soc = soc_init
        self.capacity = Q_CAP_MAX * health
        self.r0 = R_INTERNAL
        self.r1, self.c1 = 0.015, 2000
        self.r2, self.c2 = 0.030, 10000
        self.state = [soc_init, 0.0, 0.0]

    def get_ocv(self, soc):
        soc_clamped = max(soc, 0.001)
        return 3.0 + 1.0 * soc_clamped + 0.2 * np.log(soc_clamped + 0.01)

    def power_drain(self, t, p):
        # P_disp = k1 * brightness * alpha
        pd = 1.8 * p['brightness'] * p['alpha']
        # P_cpu = k2 * freq^2 * util
        pc = 3.5 * (p['cpu_freq']**2) * p['cpu_util']
        # P_net
        pn = 2.1 * p['signal'] if p['net'] else (0.5 if t < 1.5 else 0.05)
        
        total = pd + pc + pn + 0.1
        return total * 0.65 if p.get('lpm', False) else total

## 2. ODE Solver (2RC Dynamics)
We solve for SOC, and the two polarization voltages $V_{p1}, V_{p2}$.

In [ ]:
def model(y, t, bat, profile):
    soc, v1, v2 = y
    p = bat.power_drain(t, profile)
    voc = bat.get_ocv(soc)
    v_diff = voc - v1 - v2
    disc = v_diff**2 - 4 * p * bat.r0
    vt = (v_diff + np.sqrt(max(0, disc))) / 2
    i = p / vt
    
    dsoc = -i / bat.capacity
    dv1 = -v1/(bat.r1*bat.c1) + i/bat.c1
    dv2 = -v2/(bat.r2*bat.c2) + i/bat.c2
    return [dsoc, dv1, dv2]

## 3. Scenarios and Visualization

In [ ]:
t = np.linspace(0, 12*3600, 1000)
b = SmartphoneBattery()

prof_p = {'brightness': 1.0, 'alpha': 0.8, 'cpu_freq': 1.0, 'cpu_util': 0.9, 'net': True, 'signal': 1.8}
prof_l = {'brightness': 0.3, 'alpha': 0.1, 'cpu_freq': 0.4, 'cpu_util': 0.1, 'net': False, 'signal': 1.0, 'lpm': True}

res_p = odeint(model, [1.0, 0, 0], t, args=(b, prof_p))
res_l = odeint(model, [1.0, 0, 0], t, args=(b, prof_l))

plt.figure(figsize=(12, 6))
plt.plot(t/3600, res_p[:, 0]*100, label='Power User', lw=2)
plt.plot(t/3600, res_l[:, 0]*100, label='Limited User (LPM)', lw=2)
plt.title("SOC Comparison: Power vs Limited User")
plt.xlabel("Hours")
plt.ylabel("SOC (%)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()